In [ ]:
import jax
import jax.numpy as jnp
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

In [ ]:
KEY = jax.random.PRNGKey(42)
def random(*shape):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    return jax.random.normal(subkey, shape)

def symMat(dim):
    A = jnp.array(random(dim, dim) > 0.2, jnp.float32)
    return (A - A.T)


def scale_free_antisymmetric(degree, dim):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    """Build negative graph-Laplacian dynamics from a Watts-Strogatz graph."""
    if degree >= dim:
        raise ValueError(
            "small_world_degree must be smaller than dim for the small-world generator"
        )
    graph_seed = int(jax.random.bits(subkey, (), dtype=jnp.uint32))
    graph = nx.scale_free_graph(n=dim, seed=graph_seed)
    adjacency = nx.to_numpy_array(graph, dtype=np.float32)

    dense_matrix = (adjacency - adjacency.T)

    return adjacency

In [ ]:
def get_synced(systemA, systemB, sync_over):
    def foo(stateA, stateB):
        s1 = systemA(stateA)
        s2 = systemB(stateB)
        for sync in sync_over:
            path1, path2, w = sync
            s_v = s1[path1]*w + s2[path2]*(1 - w)
            s1 = s1.at[path1].set(s_v*0.01 + s1[path1]*0.99)
            s2 = s2.at[path2].set(s_v*0.01 + s2[path2]*0.99)
        return s1, s2

    return foo

In [ ]:
dim = 10
# scale = 0.1
initial = {'a': random(dim), 'b': random(dim)}

Q = scale_free_antisymmetric(2, dim)
Q1 = scale_free_antisymmetric(2, dim)

def transform_state(state):
    return jax.nn.tanh(Q @ state)

def transform_state1(state):
    return jax.nn.tanh(Q1 @ state)

px.imshow(Q).show()
hab_ind = abs(Q).sum(axis=1).argmax()
hab_ind1= abs(Q).sum(axis=1).argmax()
print(f"Most connected node: {hab_ind} with {abs(Q).sum(axis=1).max()} connections")
print(f"Most connected node1: {hab_ind1} with {abs(Q).sum(axis=1).max()} connections")
# Q = Q / dim

assert abs(initial['a'] - initial['b']).sum() != 0

In [ ]:
G = nx.from_numpy_array(Q)
G1 = nx.from_numpy_array(Q1)
pos = nx.layout.kamada_kawai_layout(G)
pos1 = nx.layout.kamada_kawai_layout(G1)

nx.draw(G, pos, node_size=1)
plt.show()
nx.draw(G1, pos1, node_size=1)
plt.show()

In [ ]:
s = get_synced(transform_state, transform_state, [[hab_ind, hab_ind, 0.5]])
scale = 0.1
s1, s2 = initial['a']*scale, initial['b']*scale
H = []
for i in range(10000):
    if i % 10 == 0:
        # print(s1, s2)
        m = jnp.mean((s1 - s2)**2)**0.5
        s1, s2 = s(s1, s2)
        H.append(m)
        # s1, s2 = transform_state(s1), transform_state(s2)

plt.plot(H)
plt.show()